In [9]:
import pickle
import geopandas as gpd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

# Load the feature matrix
wards = gpd.read_file("../../Data/floods.gpkg")

# Create elevation range feature
wards['elevation_range_m'] = wards['elevation_max_m'] - wards['elevation_min_m']

# Drop geometry and metadata for modelling — keep only numeric features and target
feature_cols = [
    'pop2009',
    'rain_cumulative_mm',
    'rain_max_daily_mm',
    'rain_preflood_7d_mm',
    'elevation_mean_m',
    'elevation_min_m',
    'elevation_max_m',
    'elevation_range_m',
    'slope_mean_deg'
]

X = wards[feature_cols]
y = wards['flooded']

print(f"Feature matrix shape : {X.shape}")
print(f"Class distribution   :\n{y.value_counts(normalize=True)}")
print(f"\nMissing values:\n{X.isnull().sum()}")

Feature matrix shape : (1450, 9)
Class distribution   :
0    0.788276
1    0.211724
Name: flooded, dtype: float64

Missing values:
pop2009                0
rain_cumulative_mm     0
rain_max_daily_mm      0
rain_preflood_7d_mm    0
elevation_mean_m       0
elevation_min_m        0
elevation_max_m        0
elevation_range_m      0
slope_mean_deg         0
dtype: int64


In [10]:
# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2026, stratify=y)
X_train.head()

,pop2009,rain_cumulative_mm,rain_max_daily_mm,rain_preflood_7d_mm,elevation_mean_m,elevation_min_m,elevation_max_m,elevation_range_m,slope_mean_deg
454,20756.0,229.351028,14.185050,27.096939,1255.182093,1164.0,1404.0,240.0,2.742085
223,25605.0,0.000000,0.000000,0.000000,1561.165994,1471.0,1630.0,159.0,4.014922
1347,26216.0,74.334179,15.210496,12.359124,270.326545,197.0,327.0,130.0,0.683518
841,27830.0,158.794473,24.094767,15.352821,1616.746939,1344.0,1972.0,628.0,4.300813
718,27731.0,137.708249,12.970549,27.609018,1903.645283,1661.0,2169.0,508.0,9.891995


In [11]:
# Scale features using StandardScaler
from sklearn.preprocessing import StandardScaler

# Instantiate the scaler and fit-transform the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using the same scaler
X_test_scaled = scaler.transform(X_test)

In [12]:
# Build a simple XGBoost model
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

# Train the model
model.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = model.predict(X_test_scaled)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       343
           1       0.73      0.62      0.67        92

    accuracy                           0.87       435
   macro avg       0.82      0.78      0.80       435
weighted avg       0.87      0.87      0.87       435



In [13]:
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV

# Create a pipeline with SMOTE, scaling, and XGBoost
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(random_state=42))
])

# Define parameters for GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1]
}

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

ValueError: Invalid parameter learning_rate for estimator Pipeline(steps=[('smote', SMOTE(random_state=42)), ('scaler', StandardScaler()),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, gamma=None, gpu_id=None,
                               importance_type='gain',
                               interaction_constraints=None, learning_rate=None,
                               max_delta_step=None, max_depth=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, n_estimators=100,
                               n_jobs=None, num_parallel_tree=None,
                               random_state=42, reg_alpha=None, reg_lambda=None,
                               scale_pos_weight=None, subsample=None,
                               tree_method=No

In [ ]:
# Handle class imbalance using SMOTE
from imblearn.over_sampling import SMOTE

# Instantiate SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Train the model on the SMOTE data
model.fit(X_train_smote, y_train_smote)

# Predict on the test set
y_pred_smote = model.predict(X_test)

# Evaluate the model
print("Classification Report after SMOTE:")
print(classification_report(y_test, y_pred_smote))


Classification Report after SMOTE:
              precision    recall  f1-score   support

           0       0.92      0.86      0.89       229
           1       0.57      0.72      0.64        61

    accuracy                           0.83       290
   macro avg       0.75      0.79      0.76       290
weighted avg       0.85      0.83      0.83       290



In [ ]:
# Use GridSearchCV to find the best hyperparameters
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring='f1', n_jobs=-1)

grid_search.fit(X_train_smote, y_train_smote)
print(f"Best parameters: {grid_search.best_params_}")

# Train the model with the best parameters
best_model = grid_search.best_estimator_

best_model.fit(X_train_smote, y_train_smote)

# Predict on the test set
y_pred_best = best_model.predict(X_test)

# Evaluate the model
print("Classification Report with Best Parameters:")
print(classification_report(y_test, y_pred_best))


/home/ngundo_larry/anaconda3/envs/learn-env/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ngundo_larry/anaconda3/envs/learn-env/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ngundo_larry/anaconda3/envs/learn-env/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2

Best parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}
Classification Report with Best Parameters:
              precision    recall  f1-score   support

           0       0.92      0.87      0.89       229
           1       0.60      0.70      0.65        61

    accuracy                           0.84       290
   macro avg       0.76      0.79      0.77       290
weighted avg       0.85      0.84      0.84       290



Saving the model:

In [ ]:
with open('../best_xg_boost_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Best XGBoost model saved")

Best XGBoost model saved
